# Theorem Audit and Traceability

**Objective.** Inspect theorem status, proof tier, code anchors, tests, and remaining non-defended rows.

**Run mode.** Analysis only. This notebook reads locked ORIUS artifacts
and does not retrain models, rewrite release manifests, or mutate runtime traces.

In [ ]:
from __future__ import annotations

import csv
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "reports").exists():
    ROOT = Path.cwd().parent
PUBLICATION = ROOT / "reports" / "publication"
SPLIT_ROOT = ROOT / "reports" / "split_training"
RELEASE_ID = (SPLIT_ROOT / "latest_release_id.txt").read_text().strip()
FREEZE = ROOT / "reports" / "predeployment_freeze" / RELEASE_ID

def read_csv(relpath: str) -> pd.DataFrame:
    path = ROOT / relpath
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_csv(path)

def read_json(relpath: str) -> dict:
    path = ROOT / relpath
    if not path.exists():
        raise FileNotFoundError(path)
    return json.loads(path.read_text())

def display_path(relpath: str) -> None:
    path = ROOT / relpath
    print(f"{relpath}: {'exists' if path.exists() else 'missing'}")

## Active theorem audit

In [ ]:
audit = read_csv("reports/publication/active_theorem_audit.csv")
audit[[
    "theorem_id",
    "title",
    "surface_kind",
    "defense_tier",
    "proof_tier",
    "code_correspondence",
    "weakest_step",
]]

In [ ]:
tier_counts = audit["defense_tier"].value_counts().rename_axis("defense_tier").reset_index(name="count")
display(tier_counts)
ax = tier_counts.plot(x="defense_tier", y="count", kind="bar", legend=False, figsize=(8, 4))
ax.set_title("Theorem defense-tier distribution")
ax.set_ylabel("Rows")
plt.tight_layout()

## Reviewer-facing theorem debt

These rows are not failures by themselves, but they must not be
described as active flagship theorems unless their proof, code, test,
artifact, and manuscript gates are promoted.

In [ ]:
debt = audit[~audit["defense_tier"].isin(["flagship_defended", "supporting_defended"])]
debt[["theorem_id", "title", "defense_tier", "scope_note", "remediation_class"]]